# Parameter Sampling with Modeldag

This notebook demonstrates how users can employ Mickael Rigault's powerful and flexible [modeldag](https://github.com/MickaelRigault/modeldag) package for statistical sampling. The `modeldag` package functions similarly to LightCurveLynx's internal sampling package, by defining a directed acyclic graph of parameters that are generated from each other. Users should be able to convert back and forth between the two. However, we provide a direct wrapper to `modeldag` in case users are more familiar with it, want to define the DAG in a single dictionary, or want to make use of the package's additional helper functions.

Note that the [modeldag](https://github.com/MickaelRigault/modeldag) is not installed as part of the default LightCurveLynx installation. Users will need to manually install pzflow via pip (e.g. `pip install modeldag`) in order to run this notebook.

## Introduction to modeldag

The `modeldag` package provides a user-friendly framework for defining directed acyclic graphs of parameters. The entire graph is specified in a dictionary where the keys are the parameters to be sampled. The values contain information need to sample. The most common formulation of a value is another dictionary that includes the following information:

  * **func**: The function that will be used to set this parameter.
  * **kwargs**: Another dictionary where the keys are the input parameters and the values are the their inputs.

For example if we want a single parameter `a` sampled from a numpy normal distribution with mean=10 and standard deviation=2, we could specify the model as:

In [1]:
import numpy as np
import modeldag

dag_dict = {
    "a": {"func": np.random.normal, "kwargs": {"loc": 10, "scale": 2}},
}
model = modeldag.ModelDAG(dag_dict)

model.draw(100).head()

,a
0,12.496716
1,11.422468
2,10.607151
3,8.847061
4,10.621678


The power of the `modeldag` package is that you can use the `@` notation to chain parameter dependencies. For example, we can define a second parameter `b` as being sampled from a distribution that depends on the value of the `a` parameter. In this case we first sample `a` and then sample `b` from a normal distribution centered on `a`.

In [2]:
dag_dict = {
    "a": {"func": np.random.normal, "kwargs": {"loc": 10, "scale": 2}},
    "b": {"func": np.random.normal, "kwargs": {"loc": "@a", "scale": 1}},
}
model = modeldag.ModelDAG(dag_dict)

model.draw(100).head()

,a,b
0,12.496716,12.602476
1,11.422468,10.467341
2,10.607151,11.917231
3,8.847061,7.802542
4,10.621678,10.386603


# The ModelDAGNode

LightCurveLynx provides a `ModelDAGNode` node to wrap `modeldag.ModelDAG`. Sampling this node samples all the parameters at once and makes them available via LightCurveLynx's dot notation.

In [3]:
from lightcurvelynx.math_nodes.modeldag_node import ModelDAGNode

node = ModelDAGNode(model, node_label="test")

state = node.sample_parameters(num_samples=5)
print(state)

test:
    a: [ 6.17061202  8.17822741  7.51472471 11.67845866  9.05540794]
    b: [ 7.9745642   8.74313719  8.79061662 12.23892074  9.804462  ]


This allows us to use the sampled parameters in chaining.

We can define a `modeldag.ModelDAG` that generates values for two parameters (a sin wave's amplitude and frequency), then feed those into a LightCurveLynx model using the dot notation.

In [ ]:
from lightcurvelynx.models.basic_models import SinWaveModel

dag_dict = {
    "amplitude": {"func": np.random.normal, "kwargs": {"loc": 10, "scale": 2}},
    "frequency": {"func": np.random.uniform, "kwargs": {"low": 0.1, "high": 1.0}},
}
dag_params_node = ModelDAGNode(modeldag.ModelDAG(dag_dict))

source = SinWaveModel(
    amplitude=dag_params_node.amplitude,
    frequency=dag_params_node.frequency,
    t0=0.0,
    ra=10.0,
    dec=-5.0,
    node_label="source",
)
state = source.sample_parameters(num_samples=5)
print(state)

SinWaveModel_0:
    ra: [10. 10. 10. 10. 10.]
    dec: [-5. -5. -5. -5. -5.]
    redshift: [None None None None None]
    t0: [0. 0. 0. 0. 0.]
    distance: [None None None None None]
    brightness: [0. 0. 0. 0. 0.]
    amplitude: [12.49671627 11.42246824 10.60715148  8.8470605  10.62167849]
    frequency: [0.79639407 0.53095556 0.27991764 0.44830119 0.50683856]
ModelDAGNode:_non_func_1:
    amplitude: [12.49671627 11.42246824 10.60715148  8.8470605  10.62167849]
    frequency: [0.79639407 0.53095556 0.27991764 0.44830119 0.50683856]


We could then use that sampling to evaluate multiple SEDs.

In [ ]:
import matplotlib.pyplot as plt

times = np.linspace(0, 10, 100)
wavelengths = np.array([500])

fluxes = source.evaluate_sed(times, wavelengths, graph_state=state)

for i in range(5):
    amp = state["source"]["amplitude"][i]
    freq = state["source"]["frequency"][i]
    label = f"Sample {i + 1}: A={amp:.2f}, f={freq:.2f}"
    plt.plot(times, fluxes[i, :, 0], label=label)
plt.xlabel("Time")
plt.ylabel("Flux")
plt.legend()
plt.show()

KeyError: 'Unknown GraphState key: test'

### GivenValueList

The `GivenValueList` node returns the values from a given list (in the order in which they are given). This is primarily used for testing:

In [ ]:
from lightcurvelynx.math_nodes.given_sampler import GivenValueList

brightness_dist = GivenValueList([18.0, 20.0, 22.0, 25.0])
model = ConstantSEDModel(brightness=brightness_dist, node_label="test")
state = model.sample_parameters(num_samples=3)
print(state["test"]["brightness"])

NameError: name 'ConstantSEDModel' is not defined

The `GivenValueList` is the only stateful parameterized node. For testing purposes if you query the node multiple times, it will give the next unsampled items from the list.

In [ ]:
state = model.sample_parameters(num_samples=1)
print(state["test"]["brightness"])

Because it is stateful, the `GivenValueList` does **not** support parallel execution. The simulation will fail with an error if a `GivenValueList` is used.

### GivenValueSampler

The `GivenValueSampler` node returns a random value (with replacement) from a given list:

In [ ]:
from lightcurvelynx.math_nodes.given_sampler import GivenValueSampler

brightness_dist = GivenValueSampler([18.0, 20.0, 22.0])
model = ConstantSEDModel(brightness=brightness_dist, node_label="test")
state = model.sample_parameters(num_samples=10)
print(state["test"]["brightness"])

The node can also take a list of weights to sample from different distributions.

In [ ]:
brightness_dist = GivenValueSampler([18.0, 20.0, 22.0], weights=[0.5, 0.3, 0.2])
model = ConstantSEDModel(brightness=brightness_dist, node_label="test")
state = model.sample_parameters(num_samples=1000)
print("Brightness 18:", np.count_nonzero(state["test"]["brightness"] == 18.0))
print("Brightness 20:", np.count_nonzero(state["test"]["brightness"] == 20.0))
print("Brightness 22:", np.count_nonzero(state["test"]["brightness"] == 22.0))

We can also provide a single integer value N to sample from the range [0, N-1].

In [ ]:
brightness_dist = GivenValueSampler(10)
model = ConstantSEDModel(brightness=brightness_dist, node_label="test")
state = model.sample_parameters(num_samples=1000)
for i in range(5):
    print(f"Brightness {i}:", np.count_nonzero(state["test"]["brightness"] == i))

This is useful when sampling indices of other lists (see the example in the "Combining Node Types" section below).

### GivenValueSelector

The `GivenValueSelector` node takes a single input parameter *index* and uses that to lookup the parameter's value from a given list. Which item is selected is determined by the `GivenValueSelector`'s *index* parameter. If we sample the *index* from a given distribution we can sample items for the list.

Below we use a constant value for *index* so we return the same element each time:

In [ ]:
from lightcurvelynx.math_nodes.given_sampler import GivenValueSelector

brightness_dist = GivenValueSelector([18.0, 20.0, 22.0], index=2)
model = ConstantSEDModel(brightness=brightness_dist, node_label="test")
state = model.sample_parameters(num_samples=10)
print(state["test"]["brightness"])

At first glance, the `GivenValueSelector` node may appear redundant with the `GivenValueSampler` node. After all, both nodes are selecting a value from a list. However, as we will see in the next section, the power of these nodes comes from when they are used together to sample consistently from multiple lists.

## Combining Node Types

We can perform complex sampling operations by combining multiple types of nodes. For example, imagine that we wanted to sample from a list of known objects where we have a list of the RAs, decs, brightness, and redshifts. We can combine a random selection of the object's index (by sampling a `index` parameter) with nodes that look up the value for that object index in each of the corresponding lists:

In [ ]:
ra_list = [10.0, 20.0, 30.0, 40.0, 50.0]
dec_list = [1.0, 2.0, 3.0, 4.0, 5.0]
brightness_list = [15.0, 16.0, 17.0, 18.0, 19.0]

index_dist = GivenValueSampler(5)  # Samples indices 0 to 4 uniformly

# Use the same sampled index for ra, dec, and brightness.
model = ConstantSEDModel(
    brightness=GivenValueSelector(brightness_list, index=index_dist),
    ra=GivenValueSelector(ra_list, index=index_dist),
    dec=GivenValueSelector(dec_list, index=index_dist),
    node_label="model",
)

state = model.sample_parameters(num_samples=10)
for i in range(10):
    ra = state["model"]["ra"][i]
    dec = state["model"]["dec"][i]
    brightness = state["model"]["brightness"][i]
    print(f"Sample {i + 1}: ({ra}, {dec}) = {brightness}")

The `GivenValueSampler` node chooses an object index value from the range [0, 5). The output of this node (the index) is passed as the input to multiple `GivenValueSelector` nodes to extract the corresponding element from each of the lists.

Any important consideration is that each node in the graph is only sampled once. This means a *single* index is chosen and used for all three lists. For each sample, the value of all parameters (RA, Dec, and brightness) will be consistent for a single object.

For other examples of how these types of nodes can be combined, see the implementation of the `MultiLightcurveTemplateModel` and the `RandomMultiObjectModel` models.

## Sampling from Tables

Instead of lists, we might want to extract values from tabular data represented as an a dictionary, AstroPy Table, or Pandas Dataframe. The `TableSampler` node will sampling a row from given tabular data and store a unique parameter for each column of the table.

For example we can create a table columns 'A', 'B', and 'C' and sample from those:

In [ ]:
from astropy.table import Table

from lightcurvelynx.math_nodes.given_sampler import TableSampler

raw_data_dict = {
    "A": [1, 2, 3, 4, 5, 6, 7, 8],
    "B": [2, 3, 4, 5, 4, 3, 2, 1],
    "C": [3, 4, 5, 6, 7, 8, 9, 10],
}
data = Table(raw_data_dict)

table_node = TableSampler(data, in_order=True, node_label="node")
state = table_node.sample_parameters(num_samples=3)
print(state)

The `in_order` flag tells the node whether to extract the rows in order (`True`) or randomly with replacement (`False`). Note that the `TableSampler` is **not** stateful. If called multiple times (with `in_order=True`), it will return the first N rows from the table each time. It will also produce a warning for the user.

In [ ]:
state = table_node.sample_parameters(num_samples=3)
print(state)

If users want to perform multiple sequential simulations using different parts of the table, then they will need to use the ``sample_offset`` parameter.

In [ ]:
state = table_node.sample_parameters(num_samples=3, sample_offset=3)
print(state)

However, most users should not need to ever set ``sample_offset`` directly. If you are running a parallelized simulation, the software sets and uses this value behind the scenes to ensure that every worker is operating on a different part of the table.

As with other node types, we can use the dot notation to use these values as input for other models. For example, let’s assume that the 'B' column corresponds to Brightness, 'A' corresponds to RA, and 'C' is not used.

In [ ]:
table_node = TableSampler(data, in_order=False, node_label="node")
model = ConstantSEDModel(
    brightness=table_node.B,
    ra=table_node.A,
    node_label="test",
)

state = model.sample_parameters(num_samples=10)
print(state)

## Statistical Distributions

### Basic (Numpy) Distributions

The `NumpyRandomFunc` node allows users to sample from most simple distributions supported by numpy, such as uniform or normal. The node's constructor takes the name of the numpy function and then any parameters as keyword arguments.

In the following cell we generate 10,000 samples from a normal distribution with mean=0.0 and standard deviation=1.0.

In [ ]:
import matplotlib.pyplot as plt

normal_dist = NumpyRandomFunc("normal", loc=0.0, scale=1.0, node_label="normal_dist")
samples = normal_dist.sample_parameters(num_samples=10000)
plt.hist(samples["normal_dist"]["function_node_result"], bins=50, density=True)
plt.title("Histogram of Samples from Normal Distribution")
plt.xlabel("Value")
plt.ylabel("Density")
plt.show()

The keyword arguments for the `NumpyRandomFunc` can be chained to take other parameters themselves. In following example, we generate an x value for each row uniformly [1, 10]. Then we use that x value to set the mean of the normal distribution for that row.

In [ ]:
x_dist = NumpyRandomFunc("uniform", low=1.0, high=10.0, node_label="mean_dist")
normal_dist = NumpyRandomFunc("normal", loc=x_dist, scale=1.0, node_label="normal_dist")
samples = normal_dist.sample_parameters(num_samples=1_000)

plt.scatter(
    samples["mean_dist"]["function_node_result"], samples["normal_dist"]["function_node_result"], alpha=0.5
)
plt.xlabel("Mean")
plt.ylabel("Value")
plt.title("Scatter Plot of Samples from Normal Distribution with Varying Mean")
plt.show()

## Combining Distributions

There are plenty of times you might want to move beyond the simple numpy sampling functions. For example, a user might want to generate numbers uniformly from the union of ranges [0, 4] U [6, 10].  We can do this by chaining nodes. Here we use a `RandomChoiceNode` (will be introduced in v0.4.2) to pick the distribution.

In [ ]:
from lightcurvelynx.math_nodes.random_choice import RandomChoiceNode

input1 = NumpyRandomFunc("uniform", low=0, high=4, node_label="input1")
input2 = NumpyRandomFunc("uniform", low=6, high=10, node_label="input2")
choice_node = RandomChoiceNode([input1, input2], node_label="choice_node")

sampled_state = choice_node.sample_parameters(num_samples=5_000)
results = sampled_state["choice_node"]["function_node_result"]

plt.hist(results, bins=50)
plt.title("Histogram of Samples from RandomChoiceNode")
plt.xlabel("Value")
plt.show()